# bsc_02 — MVP Stage 1

Gate 1 đã **PROCEED**. Chạy plan §6 bước 3→13 theo ma trận chuẩn hóa S/D/I/H.

**Notebook này cố ý MỎNG.** Mọi logic nằm ở `bsc/mvp.py` + `bsc/experiment.py` +
`bsc/atlas.py`, có test chạy trên phantom (`tests/test_mvp.py`, 11 test). Trước đây logic
nằm thẳng trong notebook nên chỉ lộ lỗi khi chạy trên Colab — sửa cách đó rồi.

| Trục | |
|---|---|
| **S** | S0 xương GT · S1 GT+jitter · S2 xương dự đoán |
| **D** | D0 oracle · D1 atlas theo fold |
| **I** | I0 mri · I1 +grad · I2 +sdf · I3 +prob |
| **H** | H0 chỉ occupancy · H1 +presence |

`P0=S0D0I0H0` · `P1=S0D0I1H0` · `P2=S0D1I2H1` · `P3=S0D1I3H1` (**P3 = M8-D**, một run)

**Thiết kế đánh giá:** train/val từ 404 ca TRAIN, fold theo `splits_zib_v1_fixed.json`
(train = fold 1–4, val = fold 0). Baseline = prediction **out-of-fold** 150ep.
**103 ca test không đụng tới.** Atlas D1 chỉ xây từ ca train của fold.

**Báo cáo (M0 §6):** không dùng ASSD tổng — mục tiêu quy về ASSD tổng chỉ 0.006mm, nhỏ
ngang sai khác giữa hai implementation metric. Dùng **mẫu số vùng mỏng** + presence F1.


### 0. Config + kiểm setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys, glob, json
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!pip install -q nibabel SimpleITK 2>/dev/null

import numpy as np
from tqdm import tqdm
from bsc import core, metrics, headroom, io_utils, atlas as atlas_mod, mvp
from bsc import experiment as X
from bsc.core import RayConfig

BSC_ROOT = "/content/drive/MyDrive/bsc"
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"
PROB_DIR = f"{BSC_ROOT}/baselines/oof_prob"
for sub in ["splits", "baselines", "atlas", "runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

cfg = RayConfig()
CART = {"femoral_cart": 2, "med_tib_cart": 4}
BONE = {"femoral_cart": 1, "med_tib_cart": 3}

ok = True
def chk(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"{'OK   ' if cond else 'THIEU'} | {name}{('  -> ' + detail) if detail else ''}")

labs = sorted(glob.glob(f"{RAW}/labelsTr/oaizib_*.nii.gz"))
chk("imagesTr", len(glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz")) > 0)
chk("labelsTr", len(labs) > 0, f"{len(labs)} nhan")

SP = None
if labs:
    g0, SP = io_utils.load_nii(labs[0])
    SP = tuple(float(x) for x in SP)
    chk("nhan co xuong+sun", {1,2,3,4}.issubset(set(int(v) for v in np.unique(g0))))
    same = np.allclose(SP, core.SPACING, atol=1e-3)
    chk("spacing", True, f"{tuple(round(x,4) for x in SP)}"
        + ("" if same else f"  (khac core.SPACING {core.SPACING} - da xu ly)"))
    bad = []
    for p in labs[::40]:
        _, s = io_utils.load_nii(p)
        if not np.allclose(s, SP, atol=1e-3):
            bad.append(os.path.basename(p))
    chk("spacing dong nhat", not bad, f"kiem {len(labs[::40])} ca")

SPLITS = f"{BSC_ROOT}/splits/splits_zib_v1_fixed.json"
chk("splits_zib_v1_fixed", os.path.exists(SPLITS))
CVP = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)
chk("CV 150ep (baseline OOF)", bool(CVP) and
    len(glob.glob(f"{CVP[0]}/fold_*/validation/oaizib_*.nii.gz")) > 0)

n_prob = len(glob.glob(f"{PROB_DIR}/**/*.nii.gz", recursive=True))
print(f"(tuy chon) OOF softmax cho P3/M8-D -> {n_prob} file"
      + ("" if n_prob else "   (chua co: P0-P2 van chay duoc)"))
print("\n" + ("=> SAN SANG" if ok else "=> THIEU NGUYEN LIEU"))

### 1. QC hình học trên xương thật — §6 bước 3, 5, 6

**Cổng dừng:** M3 < 99.5% · M4 < 90% · M2 ASSD > 0.1mm · khung suy biến > 0%.

In [ ]:
QC_CKPT, N_QC = f"{BSC_ROOT}/runs/mvp_geom_qc.jsonl", 10
cases_all = sorted(os.path.basename(p)[:-len("_0000.nii.gz")]
                   for p in glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
cases_all = [c for c in cases_all if c.startswith("oaizib_")]

done = ({(json.loads(l)["case"], json.loads(l)["cls"]) for l in open(QC_CKPT)}
        if os.path.exists(QC_CKPT) else set())
with open(QC_CKPT, "a") as fh:
    for cid in tqdm(cases_all[:N_QC], desc="geom QC"):
        gt, sp = io_utils.load_nii(f"{RAW}/labelsTr/{cid}.nii.gz")
        for c in CART:
            if (cid, c) in done: continue
            bone, cart = (gt == BONE[c]), (gt == CART[c])
            if not bone.any() or not cart.any(): continue
            sdf, verts, normals = core.bone_geometry(bone, sp, cfg)
            occ = core.occupancy_target(cart, verts, normals, cfg, sp)
            rec = core.splat_rays(occ.astype(np.float32), verts, normals, cart.shape, cfg, sp)
            pr = rec > 0.5
            fh.write(json.dumps({
                "case": cid, "cls": c,
                "m3": float(core.check_normals(sdf, verts, normals, sp)[0]),
                "m4": float(core.single_interval_ratio(occ)),
                "m2_dice": float(2*(pr & cart).sum()/(pr.sum()+cart.sum()+1e-8)),
                "m2_assd": float(metrics.assd(cart, pr, sp)),
                "degenerate": bool(atlas_mod.fit_frame(verts).is_degenerate)}) + "\n")
            fh.flush()

rows = [json.loads(l) for l in open(QC_CKPT)]
print(f"\n{'lop':<14}{'M3':>8}{'M4':>8}{'M2 Dice':>10}{'M2 ASSD':>10}{'suy bien':>11}")
for c in CART:
    r = [x for x in rows if x["cls"] == c]
    if not r: continue
    f = lambda k: float(np.mean([x[k] for x in r]))
    print(f"{c:<14}{f('m3'):>7.1%}{f('m4'):>8.1%}{f('m2_dice'):>10.3f}"
          f"{f('m2_assd'):>9.3f}mm{f('degenerate'):>10.0%}")

### 2. Split + atlas theo fold — §2.2

Atlas chỉ xây từ ca **train** của fold; `assert_no_leak` biến vi phạm §2.2 thành lỗi.
**P0/P1 dùng D0 (oracle) nên không cần atlas** — atlas chỉ cần từ P2.

In [ ]:
CLS = "femoral_cart"       # §3.1: lop DAN de debug pipeline. Doi sang med_tib_cart o buoc 11.
N_TRAIN, N_VAL, RAYS = 40, 10, 20000

fold_of = json.load(open(SPLITS))["fold_of"]
zib = [c for c in cases_all if c in fold_of]
train_ids = [c for c in zib if fold_of[c] != 0][:N_TRAIN]
val_ids   = [c for c in zib if fold_of[c] == 0][:N_VAL]
print(f"train {len(train_ids)} | val {len(val_ids)}")

CV_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)[0]
def baseline_path(cid):
    p = glob.glob(f"{CV_DIR}/fold_*/validation/{cid}.nii.gz")
    return p[0] if p else None

SRC = mvp.NiftiCaseSource(RAW, CLS, CART[CLS], BONE[CLS], baseline_path, PROB_DIR)

ATLAS_PATH = f"{BSC_ROOT}/atlas/atlas_{CLS}_fold0.npz"
if os.path.exists(ATLAS_PATH):
    z = np.load(ATLAS_PATH, allow_pickle=True)
    ATLAS = atlas_mod.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                     float(z["hi"]), tuple(z["case_ids"]), 0, None)
    print(f"Nap atlas: {len(ATLAS.case_ids)} ca")
else:
    def _al(cid): return SRC.mri(cid), SRC.bone_gt(cid), SRC.cart_gt(cid)
    ATLAS = atlas_mod.build_articular_atlas(train_ids, _al, SP, cfg, n_bins=24,
                                            min_count=3, fold=0)
    np.savez_compressed(ATLAS_PATH, prob=ATLAS.prob, n_bins=ATLAS.n_bins, lo=ATLAS.lo,
                        hi=ATLAS.hi, case_ids=np.array(ATLAS.case_ids))
    print(f"Xay atlas tu {len(ATLAS.case_ids)} ca -> {ATLAS_PATH}")

atlas_mod.assert_no_leak(ATLAS, val_ids)
print(f"assert_no_leak OK | phu {np.isfinite(ATLAS.prob).mean():.1%} o luoi, "
      f"P(khop) TB {np.nanmean(ATLAS.prob):.3f}")

### 3. M5 — tiny-set overfitting trên dữ liệu thật (§3.5, bước 8)

Trượt ⇒ có **bug** ở input/target/kiến trúc/loss. Đừng chỉnh hyperparameter.

In [ ]:
from bsc import model as M
r5 = X.from_plan("P1", CLS, seed=1)
Xa, oa, pa = mvp.build_dataset(r5, SRC, train_ids[:3], cfg, rays_per_case=2000, seed=0)
i = np.random.default_rng(0).choice(len(Xa), min(4000, len(Xa)), replace=False)

net5 = M.RayEncoder1D(in_channels=len(r5.channels), with_presence=r5.with_presence)
h5 = M.fit(net5, Xa[i], oa[i], pa[i], epochs=80, batch_size=512, lr=3e-3, seed=0)
op, _ = M.predict_rays(net5, Xa[i])
m5_dice = float(2*((op>0.5) & oa[i].astype(bool)).sum()/((op>0.5).sum()+oa[i].sum()+1e-8))
print(f"M5 loss {h5[0]['loss']:.4f} -> {h5[-1]['loss']:.4f} | occ-Dice(train) {m5_dice:.3f}")
print("CONG: Dice > 0.90")

### 4. Ma trận P0–P3 + bảng ablation M8 (§7, §8 — bước 7)

P3 tự bỏ qua nếu chưa có OOF softmax. **M8-A/M8-B không nằm trong ma trận P**
(P0/P1 dùng D0+H0) nên phải dựng riêng dưới scaffold `S0-D1-H1`.

In [ ]:
RUNS, EPOCHS = {}, 30

def run_and_log(run, tag):
    res = mvp.train_run(run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=RAYS, epochs=EPOCHS, device="cuda")
    RUNS[tag] = res
    p = f" | presence F1 {res.presence['f1']:.3f} absent-recall {res.presence['absent_recall']:.3f}" \
        if res.presence else " | (H0: khong presence)"
    print(f"{tag:<7}{run.experiment_id}")
    print(f"        {run.channels} {run.domain}/{run.heads} | occ-Dice {res.val_occ_dice:.3f}{p}")
    return res

for alias in ("P0", "P1", "P2", "P3"):
    kw = {"prob_source": X.DEFAULT_PROB_SOURCE} if alias == "P3" else {}
    run = X.from_plan(alias, CLS, seed=1, **kw)
    if "prob" in run.channels and not glob.glob(f"{PROB_DIR}/{CLS}/*.nii.gz"):
        print(f"{alias}: bo qua - chua co OOF softmax"); continue
    run_and_log(run, alias)

# --- M8: scaffold S0-D1-H1, chi doi kenh (§8) ---
for role in ("M8-A", "M8-B"):
    inp = X.M8_INPUTS[role]
    run = X.RunConfig(cls=CLS, surface="S0", domain="D1", inputs=inp, heads="H1", seed=1)
    run_and_log(run, role)

print(f"\n{'M8':<7}{'run':<8}{'kenh':<20}{'occ-Dice':>10}{'presence F1':>13}")
for tag, r in RUNS.items():
    role = r.run.m8_role
    if not role: continue
    f1 = f"{r.presence['f1']:.3f}" if r.presence else "-"
    print(f"{role:<7}{tag:<8}{str(r.run.channels):<20}{r.val_occ_dice:>10.3f}{f1:>13}")

# Dong gop coarse prior: moc DUNG la M8-A (I0=mri) vs M8-D (I3=mri+prob) - khac dung
# MOT kenh. So M8-C (I2=mri+sdf) voi M8-D se doi HAI kenh cung luc (§9 canh bao).
if "M8-A" in RUNS and "P3" in RUNS:
    d = RUNS["P3"].val_occ_dice - RUNS["M8-A"].val_occ_dice
    print(f"\nDong gop coarse ResEnc prior (M8-A -> M8-D): {d:+.3f} occ-Dice")
    print("Lon => mo hinh chu yeu HIEU CHINH ResEnc, khong tu doc MRI (§3.5 M8).")

### 5. M6 — negative control hướng tia (§3.5, bước 9)

**Test phản bác chính.** Hướng tùy ý tốt ngang pháp tuyến ⇒ lợi ích không đến từ hệ tọa độ.

In [ ]:
BEST = "P2" if "P2" in RUNS else next(iter(RUNS))
base_run = RUNS[BEST].run
m6 = {}
for mode in ("normal", "axial", "tangent", "random"):
    res = mvp.train_run(base_run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=RAYS, epochs=EPOCHS, direction=mode, device="cuda")
    m6[mode] = {"occ_dice": res.val_occ_dice,
                "presence_f1": res.presence["f1"] if res.presence else None}
    print(f"{mode:<9} occ-Dice {res.val_occ_dice:.3f}")

ctrl = max(m6[k]["occ_dice"] for k in ("axial", "tangent", "random"))
print(f"\nM6: normal {m6['normal']['occ_dice']:.3f} vs doi chung tot nhat {ctrl:.3f}")
print("CONG: normal phai HON HAN. Neu khong => phai bao cao trung thuc dieu do.")

### 6. M7 jitter + P4/P5 (§3.5, bước 10 & 12)

§4.9 đặt mốc: giữ **60–70%** hiệu năng khi chuyển sang bề mặt xương dự đoán.

In [ ]:
m7 = {}
for ds, dth in [(0.0, 0), (0.25, 5), (0.5, 10), (1.0, 15)]:
    Xb, ob, pb = mvp.build_dataset(base_run, SRC, val_ids, cfg, atlas=ATLAS,
                                   rays_per_case=RAYS, seed=1000,
                                   jitter_s_mm=ds, jitter_theta_deg=dth)
    op, pp = M.predict_rays(RUNS[BEST].net, Xb, device="cuda")
    d = float(2*((op>0.5) & ob.astype(bool)).sum()/((op>0.5).sum()+ob.sum()+1e-8))
    m7[f"{ds}mm/{dth}deg"] = d
    print(f"jitter {ds:.2f}mm/{dth:>2}deg  occ-Dice {d:.3f}")
print("Suy giam PHAI tu tu. Sup dot ngot => he toa do gion, xem lai truoc P5.\n")

BEST_I = base_run.inputs
for alias in ("P4", "P5"):
    kw = {"prob_source": X.DEFAULT_PROB_SOURCE} if BEST_I == "I3" else {}
    run = X.from_plan(alias, CLS, inputs=BEST_I, seed=1, **kw)
    if alias == "P5" and any(not SRC.has_pred(c) for c in train_ids + val_ids):
        print("P5: bo qua - co ca thieu prediction xuong"); continue
    run_and_log(run, alias)

if "P5" in RUNS:
    print(f"\nP5 giu {RUNS['P5'].val_occ_dice / RUNS[BEST].val_occ_dice:.0%} hieu nang "
          f"so voi xuong GT (§4.9 moc 60-70%)")

### 7. Đánh giá §3.7 (mẫu số vùng mỏng) + cổng Go/No-Go — bước 13

In [ ]:
EVAL_CKPT = f"{BSC_ROOT}/runs/mvp_eval_{CLS}_{BEST}.jsonl"
have = ({json.loads(l)["case"] for l in open(EVAL_CKPT)}
        if os.path.exists(EVAL_CKPT) else set())
with open(EVAL_CKPT, "a") as fh:
    for cid in tqdm(val_ids, desc="eval"):
        if cid in have: continue
        r = mvp.evaluate_case(base_run, RUNS[BEST].net, SRC, cid, cfg, ATLAS, device="cuda")
        if r:
            fh.write(json.dumps(r) + "\n"); fh.flush()

rows = [json.loads(l) for l in open(EVAL_CKPT)]
g = mvp.summarize_gate37(rows)
print(f"\nn = {g['n']} ca val (out-of-fold)")
print(f"loi bien vung mong  baseline {g['thin_err_baseline_mm']:.4f}mm -> "
      f"ray {g['thin_err_ray_mm']:.4f}mm")
print(f"cai thien tuong doi {g['rel_improve']:+.1%}  (muc tieu §3.7: >= +10%)")
print(f"cai thien tuyet doi {g['abs_improve_mm']:+.4f}mm  CI {g['ci']}")
print(f"so ca tot hon {g['n_better']}/{g['n']}")
if "presence_f1" in g:
    print(f"presence F1 {g['presence_f1']:.3f} | absent recall {g['absent_recall']:.3f}")

gate = {"1_thin_rel_improve": g.get("rel_improve"),
        "1_pass_10pct": g.get("pass_10pct"),
        "1_ci_low_positive": g.get("ci_low_positive"),
        "5_m6_normal_beats_controls": bool(m6["normal"]["occ_dice"] > ctrl),
        "m5_passed": bool(m5_dice > 0.90),
        "6_p5_retained": (RUNS["P5"].val_occ_dice / RUNS[BEST].val_occ_dice
                          if "P5" in RUNS else None)}
print("\n" + json.dumps(gate, indent=2, ensure_ascii=False))

registry = []
for tag, r in RUNS.items():
    rec = r.run.to_registry(dataset_revision="Dataset001_KneeOA",
                            extra={"tag": tag, "val_occ_dice": r.val_occ_dice,
                                   "presence": r.presence, "n_train_rays": r.n_train_rays,
                                   "n_train_cases": len(train_ids), "n_val_cases": len(val_ids),
                                   "epochs": EPOCHS, "rays_per_case": RAYS,
                                   "ray_k": cfg.k, "ray_d_min": cfg.d_min,
                                   "ray_d_max": cfg.d_max, "smooth_mm": cfg.smooth_mm,
                                   "atlas_n_cases": len(ATLAS.case_ids), "spacing": list(SP)})
    registry.append(rec)

out = {"class": CLS, "registry": registry, "m6": m6, "m7": m7,
       "gate37": {**g, **gate}, "m5_occ_dice": m5_dice}
path = f"{BSC_ROOT}/runs/MVP_stage1_{CLS}.json"
json.dump(out, open(path, "w"), indent=2, ensure_ascii=False)
print("\nDa ghi", path)
print("""
NHAC LAI (M0 §6): KHONG bao cao ASSD tong. Bao cao loi bien VUNG MONG, presence F1,
absent recall. M6 la test phan bac - khong co no thi ket qua duong tinh KHONG quy duoc
cho he toa do.
Buoc 11 (§6): doi CLS = "med_tib_cart" o muc 2 roi chay lai tu muc 2.""")